In [ ]:
from theia.simulation.logging import LogLoader


loader = LogLoader("log.json")
# pcl_sensors = loader.blue_pcl_sensors

In [ ]:
loader.blue_pcl_detections

In [ ]:
import pandas as pd


# pd.DataFrame(
#     [
#         {"id": sensor.id, "Rx ID": sensor.receiver.id, "Tx ID": sensor.transmitter.id}
#         for sensor in loader.blue_pcl_sensors
#     ]
# )

In [ ]:
from theia.data_loading import load_bakom_ukw_transmitters


txs = load_bakom_ukw_transmitters()
erps = [tx.erp for tx in txs]

In [ ]:
len(txs)

In [ ]:
from matplotlib import pyplot as plt


fig, ax = plt.subplots()
ax.boxplot(erps)
ax.set_ylim([0, 1000])
ax.grid(True)

In [ ]:
import datetime
from theia.data_loading import load_bakom_ukw_transmitters
from theia.grids import LatLonTerrainGrid
from theia.test_data import build_single_target_from_Bodensee

grid = LatLonTerrainGrid(
    lat_start=46.60794,
    lat_stop=47.01023,
    lat_res=0.005,
    lon_start=7.81442,
    lon_stop=9.21230,
    lon_res=0.005,
)

trajectory = build_single_target_from_Bodensee()
target = trajectory(datetime.datetime(year=2026, month=4, day=29, minute=6, second=48))
# sensor = pcl_sensors[0]
# transmitters = [sensor.transmitter for sensor in pcl_sensors]
transmitters = load_bakom_ukw_transmitters(start_id=1)
# transmitters = [tx for tx in transmitters if grid.contains(tx.point)]
high_power_transmitters = [tx for tx in transmitters if tx.erp > 500]

In [ ]:
from theia.detection.pcl import PclDetector, pcl_track_init_update_masks
from theia.grids import LatLonHeightGrid
from theia.test_data import build_pcl_receiver
from theia.types import PclMeasurementModel, PclSensor, Point
from theia.data_loading import load_bakom_ukw_transmitters
from theia.terrain import elevationAt


grid = LatLonHeightGrid(
    lat_start=46.89586,
    lat_stop=47.11407,
    lat_res=0.01,
    lon_start=8.15612,
    lon_stop=8.61887,
    lon_res=0.01,
    height_start=1000.0,
    height_res=1.0,
    height_stop=1000.0,
)

detector = PclDetector()

transmitters = load_bakom_ukw_transmitters(start_id=1)
tx_ids = [943, 1151, 921, 1113]
txs = [tx for tx in transmitters if tx.id in tx_ids]

In [ ]:
import itertools

import numpy as np


def rx_cost(rx_latlon, txs) -> float:
    rx_lat, rx_lon = rx_latlon
    point = Point(
        lat=rx_lat,
        lon=rx_lon,
        alt=elevationAt(rx_lat, rx_lon),
    )
    pcl_rx = build_pcl_receiver(rx_id=1, point=point)

    sensors = [
        PclSensor(
            id=i,
            transmitter=tx,
            receiver=pcl_rx,
            error_model=PclMeasurementModel(),
        )
        for i, tx in enumerate(txs)
    ]

    min_detectable_rcs_grids = [
        detector.minimum_detectable_rcs_grid(
            sensor.receiver,
            sensor.transmitter,
            grid,
        )[:, :, 0]
        for sensor in sensors
    ]
    n_cells = (
        min_detectable_rcs_grids[0].shape[0] * min_detectable_rcs_grids[0].shape[1]
    )
    loss = (
        np.nanmax(np.stack(min_detectable_rcs_grids, axis=-1), axis=-1).sum() / n_cells
    )

    return loss if not np.isnan(loss) else 100_000_000


def multi_rx_cost(rx_positions, txs) -> float:
    n_rxs = len(rx_positions) // 2
    sensors: list[PclSensor] = []
    for i, (rx_lat, rx_lon) in enumerate(itertools.batched(rx_positions, 2)):
        point = Point(
            lat=rx_lat,
            lon=rx_lon,
            alt=elevationAt(rx_lat, rx_lon),
        )
        pcl_rx = build_pcl_receiver(rx_id=i, point=point)

        sensors.extend(
            [
                PclSensor(
                    id=i * n_rxs + j,
                    transmitter=tx,
                    receiver=pcl_rx,
                    error_model=PclMeasurementModel(),
                )
                for j, tx in enumerate(txs)
            ]
        )

    min_detectable_rcs_grids = [
        detector.minimum_detectable_rcs_grid(
            sensor.receiver,
            sensor.transmitter,
            grid,
        )[:, :, 0]
        for sensor in sensors
    ]
    n_cells = (
        min_detectable_rcs_grids[0].shape[0] * min_detectable_rcs_grids[0].shape[1]
    )
    loss = (
        np.nanmax(np.stack(min_detectable_rcs_grids, axis=-1), axis=-1).sum() / n_cells
    )

    return loss if not np.isnan(loss) else 100_000_000

In [ ]:
rx_lat = 47.0823
rx_lon = 8.3465

rx_cost((rx_lat, rx_lon), txs)

from scipy.optimize import minimize

from theia.types import Transmitter


res = minimize(
    lambda rx_pos: multi_rx_cost(rx_pos, txs),
    (grid.lat_start, grid.lon_start, grid.lat_stop - 0.001, grid.lon_stop - 0.001),
    bounds=(
        (grid.lat_start, grid.lat_stop),
        (grid.lon_start, grid.lon_stop),
        (grid.lat_start, grid.lat_stop),
        (grid.lon_start, grid.lon_stop),
    ),
    options={
        "maxiter": 100,
    },
)

In [ ]:
res.x

In [ ]:
from theia.util import mask_to_polygon

rxs_optimized = []
for rx_lat, rx_lon in itertools.batched(res.x, 2):
    point = Point(
        lat=rx_lat,
        lon=rx_lon,
        alt=elevationAt(rx_lat, rx_lon),
    )
    pcl_rx = build_pcl_receiver(rx_id=1, point=point)
    rxs_optimized.append(pcl_rx)


sensors = [
    PclSensor(
        id=i,
        transmitter=tx,
        receiver=rx,
        error_model=PclMeasurementModel(),
    )
    for i, (rx, tx) in enumerate(itertools.product(rxs_optimized, txs))
]

init_mask, update_mask = pcl_track_init_update_masks(detector, sensors, grid, 1.0)
polygons_init = mask_to_polygon(
    init_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
polygons_update = mask_to_polygon(
    update_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)

In [ ]:
res.x

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(update_mask)

In [ ]:
import shapely

from theia.simulation.server import GeoJSONFeature, GeoJSONMultiPolygon


GeoJSONFeature(
    geometry=GeoJSONMultiPolygon.from_shapely(shapely.MultiPolygon(polygons_update))
).model_dump_json()

In [ ]:
shapely.MultiPolygon(polygons_init)

In [ ]:
shapely.MultiPolygon(polygons_update)

In [ ]:
from theia.distance import line_of_sight_distance
from theia.snr import calculate_snr
from theia.types import ConstantRcsModel, Target, Velocity, Point
from theia.util import frequency_to_wavelength, get_clear_sky_attenuation
from theia.grids import LatLonHeightGrid
from theia.detection.pcl import PclDetector, calculate_antenna_pattern

from theia.test_data import build_pcl_receiver
from theia.types import PclMeasurementModel, PclSensor, Point
from theia.data_loading import load_bakom_ukw_transmitters
from theia.terrain import elevationAt


transmitters = load_bakom_ukw_transmitters(start_id=1)
# transmitters = [tx for tx in transmitters if grid.contains(tx.point)]
high_power_transmitters = [tx for tx in transmitters if tx.erp > 250]
tx = next(tx for tx in high_power_transmitters if tx.id == 106)
rx = 47.38299
rx = 8.03427
point = Point(
    lat=rx,
    lon=rx,
    alt=elevationAt(rx, rx),
)
pcl_rx = build_pcl_receiver(rx_id=1, point=point)
sensor = PclSensor(
    id=0,
    transmitter=tx,
    receiver=pcl_rx,
    error_model=PclMeasurementModel(),
)

grid = LatLonHeightGrid(
    lat_start=46.9173,
    lat_stop=47.7428,
    lat_res=0.005,
    lon_start=7.2288,
    lon_stop=8.4040,
    lon_res=0.005,
    height_start=2000.0,
    height_stop=2000.0,
    height_res=1.0,
)

detector = PclDetector()

tgt_pos = grid.points[0]

tgt = Target(
    id=0,
    point=Point(lat=tgt_pos[0], lon=tgt_pos[1], alt=tgt_pos[2]),
    cross_section_model=ConstantRcsModel(rcs=1.0),
    velocity=Velocity(vx=0, vy=0, vz=0),
)

detector.calculate_raw_measurement(sensor.receiver, sensor.transmitter, tgt)

# calculate_snr(
#     frequency_to_wavelength(sensor.transmitter.frequency),
#     sensor.transmitter.antenna_gain,
#     sensor.receiver.antenna_gain,
#     1.0,
#     line_of_sight_distance(*sensor.transmitter.point.as_tuple(), *tgt_pos),
#     line_of_sight_distance(*sensor.receiver.point.as_tuple(), *tgt_pos),
#     sensor.transmitter.power,
#     sensor.receiver.bandwidth,
#     1,
#     300,
#     0.0,
#     0.0,

# )

snr = calculate_snr(
    wavelength=frequency_to_wavelength(tx),
    antenna_gain_transmitter=tx.antenna_gain,
    antenna_gain_receiver=rx.antenna_gain(tx.frequency),
    radar_cross_section=1.0,
    distance_transmitter_target=r_t,
    distance_receiver_target=r_r,pcl_rx_lon
    transmission_power=tx.power,
    bandwidth=rx.bandwidth,
    cpi_pulses=rx.cpi_pulses,
    equivalent_temperature=rx.noise_temperature,
    L_t=0.0,
    L_r=rx.noise_figure,
    L_a=get_clear_sky_attenuation(tx.frequency) * (r_r + r_t) / 1000.0,
    polarization_factor=0,
    pattern_propagation_factor_transmitter=calculate_antenna_pattern(
        tx, tgt.point
    ),
    pattern_propagation_factor_receiver=calculate_antenna_pattern(
        rx, tgt.point
    ),
    is_one_way=False,
)

In [ ]:
import numpy as np


20 * np.log10(48310), 20 * np.log10(80105)

In [ ]:
detector.minimum_detectable_rcs_grid(
    sensor.receiver,
    sensor.transmitter,
    grid,
)

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST
from theia.coverage import calculate_coverage
from theia.radar_equation import calculate_maximum_monostatic_range
from theia.terrain import elevationAt
from theia.test_data import build_flores_monostatic_radar
from theia.types import Point


radar_lat = POSITIONS_OF_INTEREST["Uetliberg"]["lat"]
radar_lon = POSITIONS_OF_INTEREST["Uetliberg"]["lon"]
point = Point(lat=radar_lat, lon=radar_lon, alt=elevationAt(radar_lat, radar_lon))
monostatic_radar = build_flores_monostatic_radar(point, 0, 0, 0)
coverage = calculate_coverage(
    monostatic_radar.receiver.point,
    calculate_maximum_monostatic_range(monostatic_radar),
    2000,
)

In [ ]:
max([tx.erp for tx in load_bakom_ukw_transmitters(start_id=1)])

In [ ]:
import folium

from theia.mapping import RadarMap
from theia.data_loading import load_bakom_ukw_transmitters


transmitters = load_bakom_ukw_transmitters(start_id=1)
high_power_transmitters = [tx for tx in transmitters if tx.erp > 1_000]

map = RadarMap(polygons={"coverage": coverage}).to_map()
tx_ids = [106, 272, 1132, 1151]
for tx in high_power_transmitters:
    folium.Marker(
        (tx.lat, tx.lon),
        icon=folium.Icon(color="red" if tx.id in tx_ids else "blue"),
        tooltip=f"Tx ID = {tx.id}",
    ).add_to(map)

rx = 47.38299
rx = 8.03427

folium.Marker(
    (rx, rx),
    icon=folium.Icon(color="green"),
    tooltip="Rx",
).add_to(map)

for polygon in polygons:
    folium.GeoJson(polygon).add_to(map)

folium.GeoJson(grid.get_bbox_polygon()).add_to(map)

map.save("transmitters_at_least_ERP_1kW.html")
map

In [ ]:
import numpy as np

from theia.line_of_sight import has_line_of_sight
from theia.types import Point

grid_points = grid.points_local_maxima

# Select potential positions for the receiver with line-of-sight to at least
# one transmitter.
points_with_los_to_tx = []
for point in grid_points:
    for tx in transmitters:
        has_los = has_line_of_sight(
            Point(lat=point[0], lon=point[1], alt=point[2]),
            Point(
                lat=tx.point.lat,
                lon=tx.point.lon,
                alt=tx.point.alt + tx.antenna_height,
            ),
            30.0,
        )
        if has_los:
            points_with_los_to_tx.append(point)
            break

In [ ]:
timestamps = np.arange(
    trajectory.times[0].timestamp(),
    trajectory.times[-1].timestamp(),
    4,
)
trajectory_points = [
    trajectory(datetime.datetime.fromtimestamp(t)).point for t in timestamps
]

In [ ]:
n_los = []
for p_rx in points_with_los_to_tx:
    p_rx = Point(lat=p_rx[0], lon=p_rx[1], alt=p_rx[2])
    n_los.append(sum([has_line_of_sight(p_rx, p, 30) for p in trajectory_points]))

sorted_indices = np.argsort(n_los)

In [ ]:
sorted_indices[:5]

In [ ]:
import folium

from theia.mapping import RadarMap


map = RadarMap().to_map()
# for point in points_with_los_to_tx[:20]:
#     folium.Marker((point[0], point[1])).add_to(map)
# folium.Marker((p_rx.lat, p_rx.lon), icon=folium.Icon(color="red")).add_to(map)

for p_rx in np.array(points_with_los_to_tx)[sorted_indices[:5], :]:
    folium.Marker((p_rx[0], p_rx[1]), icon=folium.Icon(color="red")).add_to(map)

for tx in transmitters:
    folium.Marker((tx.lat, tx.lon)).add_to(map)

folium.GeoJson(grid.get_bbox_polygon()).add_to(map)
map

In [ ]:
import numpy as np


In [ ]:
from theia.detection.pcl import calculate_minimum_detectable_rcs


calculate_minimum_detectable_rcs()

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

from theia.line_of_sight import has_line_of_sight
from theia.types import Point, Trajectory


def plot_has_line_of_sight(trajectory: Trajectory, observer: Point):
    timestamps = np.arange(
        trajectory.times[0].timestamp(),
        trajectory.times[-1].timestamp(),
        0.5,
    )
    has_los = []
    for t in timestamps:
        target_pos = trajectory(datetime.datetime.fromtimestamp(t)).point
        has_los.append(has_line_of_sight(observer, target_pos, 30))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(timestamps, has_los)

    return fig, ax

In [ ]:
for sensor in pcl_sensors:
    fig, ax = plot_has_line_of_sight(trajectory, sensor.receiver.point)
    ax.set_title(f"Sensor ID = {sensor.id}")

In [ ]:
import numpy as np

from theia.detection.pcl import PclDetector

rng = np.random.Generator(np.random.PCG64(seed=489299))

detector = PclDetector()
detector.calculate_pcl_detection(rng, pcl_sensors[0], target)

In [ ]:
from theia.grids import LatLonHeightGrid


grid = LatLonHeightGrid.model_validate(
    {
        "height_res": 1,
        "height_start": 1000,
        "height_stop": 1000,
        "lat_res": 0.01,
        "lat_start": 46.28243,
        "lat_stop": 47.06638,
        "lon_res": 0.01,
        "lon_start": 8.0854,
        "lon_stop": 8.84444,
    }
)

In [ ]:
from theia.detection.pcl import pcl_track_init_update_masks


init_mask, update_mask = pcl_track_init_update_masks(
    detector,
    [pcl_sensors[0]],
    grid,
    1.0,
)

In [ ]:
from theia.plotting import plot_profile

plot_profile(sensor.receiver.point, target.point, "Rx", "Target")

In [ ]:
for sensor in pcl_sensors:
    fig, ax = plot_profile(sensor.transmitter.point, target.point, "Tx", "Target")

In [ ]:
from theia.test_data import build_single_target_from_Bodensee


traj = build_single_target_from_Bodensee()
traj

In [ ]:
map = RadarMap(
    trajectories={"Trajectory": traj},
    polygons={"roi": grid.get_bbox_polygon()},
).to_map()
for poi in pois:
    folium.Marker((poi.lat, poi.lon), tooltip=poi.label).add_to(map)
map

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST
from theia.types import Point


center = Point(
    lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
    lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
    alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
)

start_point = Point(lat=47.2757, lon=8.3358, alt=1000)

In [ ]:
import datetime

import numpy as np
import shapely

from theia.maneuvers import EightManeuver

t_start = datetime.datetime(year=2026, month=4, day=28)

# maneuver = ConstantSpeedCurveManeuver(
#     center_point=center,
#     speed=300.0,
#     angular_arclength=-np.pi,
# )
maneuver = EightManeuver(
    center_point1=center,
    azimuth_center1_to_center2=np.pi / 2 / 3,
    elevation_center1_to_center2=0.0,
    speed=300.0,
)

times, points = maneuver.get_waypoints(start_time=t_start, start_pos=start_point)
curve = shapely.geometry.LineString([(p.lon, p.lat) for p in points + [points[-1]]])

In [ ]:
from theia.coordinates import CoordinateTransformations


p_center_ecef = np.array(
    CoordinateTransformations.geodetic_to_cartesian(
        center.lat,
        center.lon,
        center.alt,
    )
)
p_end_ecef = np.array(
    CoordinateTransformations.geodetic_to_cartesian(
        points[-1].lat,
        points[-1].lon,
        points[-1].alt,
    )
)
p_center2_ecef = 2 * p_end_ecef - p_center_ecef
lat, lon, alt = CoordinateTransformations.cartesian_to_geodetic(
    p_center2_ecef[0],
    p_center2_ecef[1],
    p_center2_ecef[2],
)
p_center2 = Point(lat=lat, lon=lon, alt=alt)

In [ ]:
timestamps = [t.timestamp() for t in times]
points_ecef = np.array(
    [CoordinateTransformations.geodetic_to_cartesian(*p.as_tuple()) for p in points]
)

In [ ]:
from matplotlib import pyplot as plt
from scipy.interpolate import CubicSpline

f_ecef = CubicSpline(timestamps, points_ecef, extrapolate=True)
v_ecef = f_ecef.derivative()
a_ecef = v_ecef.derivative()
t_eval = np.arange(timestamps[0], timestamps[-1], 0.1)
points_interpolated = f_ecef(t_eval)
speed_interpolated = np.linalg.norm(v_ecef(t_eval), axis=1)
accel_interpolated = np.linalg.norm(a_ecef(t_eval), axis=1)

fig, ax = plt.subplots()

ax.plot(t_eval, speed_interpolated)
# ax.set_ylim(299.995, 300.005)

In [ ]:
from theia.mapping import RadarMap
import folium


map = RadarMap().to_map()
map.location = (center.lat, center.lon)
folium.Marker((center.lat, center.lon), tooltip="Center").add_to(map)
# folium.Marker((47.34885858914191, 8.872821498768166), tooltip="Center 2").add_to(map)
folium.Marker((start_point.lat, start_point.lon), tooltip="Start point").add_to(map)
folium.GeoJson(curve).add_to(map)
map